<a href="https://colab.research.google.com/github/juanpajedrez/learn_rag_Huggingface/blob/main/Langchain_and_Unstructured_Data_Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries and OpenAI API

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Change directory to this folder
%cd /content/drive/MyDrive/ai_agents_zero_to_mastery/RAG Bootcamp/RAG/Unstructured Data

/content/drive/MyDrive/ai_agents_zero_to_mastery/RAG Bootcamp/RAG/Unstructured Data


In [ ]:
# Import the userdata module from Google Colab
from google.colab import userdata
# Retrieve the API key stored under 'genai_course' from Colab's userdata
api_key = userdata.get('ai_agent_openai')

In [ ]:
# Install libraries
!pip install langchain-community unstructured langchain-openai faiss-cpu -q
# Excel
!pip install msoffcrypto-tool -q
# Word
!pip install python-docx -q
!pip install --upgrade nltk -q
# PPT
!pip install python-pptx -q
# EPUB
!pip install pypandoc pandoc -q

# PDF
!pip install pymupdf pdfminer.six pillow_heif unstructured_inference unstructured_pytesseract pi_heif pdf2image -q
!apt-get install poppler-utils -q
!apt install tesseract-ocr -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 30.3 MB/s eta 0:00:00
   ━━━

# Configurations

In [ ]:
# --- GLOBAL SETTINGS ---
# Embedding model
EMBEDDING_MODEL = "text-embedding-3-small"

# GPT model
GPT_MODEL = "gpt-5.4-nano"

In [ ]:
# set the api key in os
import os
os.environ["OPENAI_API_KEY"] = api_key

# Helpful functions

In [ ]:
# Import essential libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from IPython.display import Markdown, display

In [ ]:
# Prepare excel data
def prepare_excel(file_path):
  df = pd.read_excel(file_path)

  # One document / chunk per row
  docs_chunks = [
      Document(page_content = "\n".join(f"{col}: {val}" for col, val in row.items()),
              metadata = {"row_index": int(i)}
              )
      for i, row in df.iterrows()
  ]
  return docs_chunks

In [ ]:
# Prepare function for retrieval
def retrieval(
    docs_chunks_passed,
    embedding_model = EMBEDDING_MODEL,
    chunk_size_passed = 800,
    chunk_size_overlap_passed = 400):

  # Split into chunks
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size_passed,
      chunk_overlap = chunk_size_overlap_passed
  )
  chunks = text_splitter.split_documents(docs_chunks_passed)

  # Load the embedding model
  embeddings = OpenAIEmbeddings(model = EMBEDDING_MODEL)

  # Create the vector store
  db_faiss = FAISS.from_documents(docs_chunks_passed, embeddings)

  return db_faiss

In [ ]:
def generation(db_faiss, query_passed, k = 20, model = GPT_MODEL):
  # Retrieve the context
  retrieved_docs = db_faiss.similarity_search_with_score(query_passed, k=k)
  context = "\n".join([doc.page_content for doc, score in retrieved_docs])

  # Build the prompt
  prompt = f"""
  answer the query {query_passed} based on the retrieved context: {context}.
  If you don't have the contet necessary, say that you do not know.
  """

  # Call the API
  model = ChatOpenAI(model = GPT_MODEL)
  answer = model.invoke(prompt)

  return answer

# Excel

In [ ]:
# Import libraries
import pandas as pd
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain_core.documents import Document

In [ ]:
# Import the data
df = pd.read_excel("Reviews.xlsx")
df.head()

,"Before you go any further though, we have a quick favor to ask...\n\nCould you take *60 seconds* to let us know what you think of the course so far (in exchange for this cute cat pic)?",Our suspicions have been confirmed: *you are *_*awesome*_*!*\n\nStart by selecting the course you're taking below.,How would you rate this course so far ⭐,What do you _*like*_ _*most*_ about the course so far?,How can the course be _*improved*_?,*Anything else* you want to tell us about your experience?
0,"Sure, I'd love to help!",Business Analytics Bootcamp (with Python): Zer...,5,NaN,NaN,NaN
1,"Sure, I'd love to help!",Business Analytics Bootcamp (with Python): Zer...,5,T,NaN,NaN
2,"Sure, I'd love to help!",Business Analytics Bootcamp (with Python): Zer...,4,Great for beginners,Focus more on topics asked in interviews,NaN
3,"Sure, I'd love to help!",Business Analytics Bootcamp (with Python): Zer...,5,The content structure,More test scenarios,I love the flow and pace of the course
4,"Sure, I'd love to help!",Business Analytics Bootcamp (with Python): Zer...,5,NaN,NaN,NaN


In [ ]:
# Test the functions
query_passed = "What are the most critical/negative reviews?"
chunks_excel = prepare_excel("Reviews.xlsx")
db_faiss = retrieval(chunks_excel, embedding_model=EMBEDDING_MODEL)
answer = generation(db_faiss, query_passed, k = 20, model = GPT_MODEL)

In [ ]:
display(Markdown(answer.content))

Based on the retrieved reviews, the most critical/negative feedback is:

- **“Statistics with Python: Zero to Mastery” (⭐ 3)**:  
  - The **code presentation isn’t well-organized**.  
  - **Google Colab (in Google Drive) is unclear** for following the code.  
  - The instructor **jumps between code pieces without explaining everything clearly**, making the reviewer unsure whether to continue.  
  - Requested improvements: **clearer presentation**, **better software/environment to show code**, and **better explanations**.

- **“Statistics with Python: Zero to Mastery” (⭐ 4)**:  
  - **Challenges are difficult for a beginner**, which **discouraged** the reviewer (though the step-by-step instructor guidance helped).

- **“Statistics with Python: Zero to Mastery” (⭐ 4)**:  
  - Would like **more real-life scenarios**.

- **“Business Analytics Bootcamp (with Python): Zero to Mastery” (lower/neutral critiques)**:  
  - One review mentions wanting **more test scenarios** (otherwise “love the flow and pace,” “practical and interesting”).

### Word

In [ ]:
# Import necessary libraries for word documents
import nltk
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
def prepare_word(file_path):
  # Load the word document into docs
  loader = UnstructuredWordDocumentLoader(file_path)
  docs_word = loader.load()
  return docs_word

In [ ]:
query_passed = "What is the main point of this document?"

# Load the word document into docs
docs_word = prepare_word("slide style design.docx")

# Apply the retrieval (text splitting and get database vectorstore for similarity search)
db_faiss = retrieval(docs_word, embedding_model=EMBEDDING_MODEL)

# Apply generation
answer = generation(db_faiss, query_passed, k = 20, model = GPT_MODEL)

In [ ]:
display(Markdown(answer.content))

The main point of this document is to provide a strict, mobile-friendly guide for turning text into effective slide prompts—emphasizing one clear idea per slide, visual-first design (slides support narration, not copy it), specific color rules for key vs. standard slides, and a required prompt template and deck structure.

### ppt

In [ ]:
# Essential class
from langchain_community.document_loaders import UnstructuredPowerPointLoader

In [ ]:
# Build a function to prepare the data
def prepare_ppt(file_path):
  loader = UnstructuredPowerPointLoader(file_path)
  docs_ppt = loader.load()
  return docs_ppt

docs_ppt = prepare_ppt("6.12 Working with PowerPoint Presentations Using Langchain.pptx")

In [ ]:
# Prepare the VDB
db_faiss_ppt = retrieval(docs_ppt, chunk_size_passed=200, chunk_size_overlap_passed=20)

In [ ]:
# Generate the answer
answer = generation(db_faiss_ppt, "What is this lecture about?")

In [ ]:
display(Markdown(answer.content))

This lecture is about using LangChain to process PowerPoint (.pptx) presentations and make their content easily searchable. It covers loading a PPT, splitting slide text into chunks, generating embeddings, storing them in a vector database (FAISS), and applying best practices for chunk size and overlap to support effective querying of slide information.

## EPUB

In [ ]:
# Import essential module
from langchain_community.document_loaders import UnstructuredEPubLoader
import pypandoc
pypandoc.download_pandoc()

In [ ]:
# Build the function
def prepare_epub(file_path):
  loader = UnstructuredEPubLoader(file_path)
  docs_epub = loader.load()
  return docs_epub

docs_epub = prepare_epub("fitzgerald-great-gatsby.epub")

In [ ]:
# Prepare the db
db_faiss_epub = retrieval(docs_epub)

In [ ]:
# Answer the query
answer = generation(db_faiss_epub, "What is the main point of this book?")

In [ ]:
display(Markdown(answer.content))

The main point of *The Great Gatsby* is that the American dream—especially the idea that you can reinvent yourself and “repeat” the past to achieve perfect happiness—ultimately fails when it’s built on illusion, wealth, and moral emptiness. Gatsby’s desperate hope to regain Daisy and the life he imagines destroys him, and the novel shows how the rich and powerful can cause ruin for others while staying protected from consequences.

## PDF

In [ ]:
# Import modules
from langchain_community.document_loaders import UnstructuredPDFLoader
from pdfminer import psparser

In [ ]:
# Build function for the PDF
def prepare_pdf(file_path):
  loader = UnstructuredPDFLoader(file_path, language = "EN")
  docs = loader.load()
  return docs

In [ ]:
docs_pdf = prepare_pdf("airbnb-original-deck-2008.pdf")

In [ ]:
# Build de DB
db_faiss_pdf = retrieval(docs_pdf)

In [ ]:
# Generate the answer
answer = generation(db_faiss_pdf, "Why is Airbnb a good investmen?")

In [ ]:
display(Markdown(answer.content))

Based on the retrieved context, Airbnb (then presented as “AirBed & Breakfast”) is a good investment for these reasons:

- **Clear customer problem it solves:** online travelers worry about **price**, and hotels can leave people “disconnected from the city and its culture.” Airbnb offers a **cheaper, more local** alternative (“Stay with a local when traveling”).
- **Strong market demand and size:** the context cites large **worldwide trip volume** and an available market share for **budget/online trips**, implying a sizable growth opportunity.
- **Simple, scalable business model:** Airbnb takes a **10% commission on each transaction**, creating a revenue stream tied directly to booking activity.
- **Differentiated product and user experience:** it enables **search by city**, **review listings**, and **booking in a few clicks**, which helps convert demand into transactions.
- **Competitive advantages vs. alternatives:** compared with offline options (e.g., Craigslist/manual rentals) and other platforms (e.g., hostels/couch-surfing), the context highlights advantages like **easy listing once**, **better matching/search (price/location)**, and a more structured **check-in/check-out + booking flow**.
- **Validated traction signals:** the context includes testimonials like “easy to use” and “made me money” plus adoption/partner strategies (e.g., events, Craigslist partnerships), suggesting real willingness to use the platform.

If you share the “retrieved context” you used for Airbnb’s later-stage financials (revenue growth, occupancy, profitability, regulation, etc.), I can tailor the investment thesis to those specifics too.